# 20 — Federating Existing MCP Servers and Discovery

Not every MCP server you need to govern was created inside API Management.
[Connect and govern an existing MCP server](https://learn.microsoft.com/en-us/azure/api-management/expose-existing-mcp-server)
covers proxying a tool server hosted *outside* API Management -- a
"passthrough" MCP server -- through the same gateway, with the same
policies and observability as notebook 19.

Typical scenarios:

- Proxy LangChain/LangServe tool servers with per-server auth and rate
  limits.
- Securely expose Azure Logic Apps- or Azure Functions-based tools with IP
  filtering and OAuth.
- Centralize MCP servers built on different runtimes into one enterprise
  catalog.
- Let GitHub Copilot, Claude, or ChatGPT reach tools across your
  organization through one governed front door.

## Portal steps

1. In your API Management instance, under **APIs**, select
   **MCP servers > + Create MCP server**.
2. Choose **Expose an existing MCP server**.
3. Under **Backend MCP server**, enter the server's base URL (for example,
   `https://learn.microsoft.com/api/mcp` for the public Microsoft Learn MCP
   server) and its **Transport type** (Streamable HTTP or SSE).
4. Fill in the new MCP server's display name/name/description and optional
   product associations, same as notebook 17.
5. Select **Create**.

### Limitations

- The external server must speak MCP `2025-06-18` or later, with either no
  authorization or a spec-compliant authorization scheme.
- For passthrough servers, API Management supports MCP **tools and
  resources** -- still no **prompts**.
- Workspaces aren't supported for MCP servers, same as REST-API-backed ones.

## Managing MCP servers as code

Every MCP server -- REST-API-backed or passthrough -- is an Azure Resource
Manager `Microsoft.ApiManagement/service/apis` resource with
`properties.type = "mcp"`. Manage it with the REST API, ARM templates,
Bicep, the Azure CLI, or Terraform, using API Management REST API version
**`2025-09-01-preview`** or later
([Manage MCP servers programmatically](https://learn.microsoft.com/en-us/azure/api-management/manage-mcp-servers-rest-api)).

Tools are a separate `apiTool` sub-resource, so CI/CD pipelines can add,
rename, or remove tools without recreating the whole MCP server.

In [ ]:
# Illustrative only: listing MCP servers (APIs of type "mcp") via the ARM
# management API, using the same DefaultAzureCredential chain as the rest
# of this course. Requires `httpx` (already a transitive dependency of the
# mcp package) and a real subscription/resource group/APIM name to run.
import httpx
from azure.identity import DefaultAzureCredential

subscription_id = "<subscription-id>"
resource_group = "<resource-group>"
apim_name = "<apim-service-name>"
api_version = "2025-09-01-preview"

credential = DefaultAzureCredential()
token = credential.get_token("https://management.azure.com/.default").token

base_url = (
    f"https://management.azure.com/subscriptions/{subscription_id}"
    f"/resourceGroups/{resource_group}/providers/Microsoft.ApiManagement"
    f"/service/{apim_name}"
)

async def list_mcp_servers():
    async with httpx.AsyncClient() as client:
        response = await client.get(
            f"{base_url}/apis",
            params={"api-version": api_version, "$filter": "type eq 'mcp'"},
            headers={"Authorization": f"Bearer {token}"},
        )
        response.raise_for_status()
        return response.json()

# Uncomment once the placeholders above are filled in:
# servers = await list_mcp_servers()
# for server in servers["value"]:
#     print(server["name"], "->", server["properties"]["displayName"])